# ML System Design: Ad Click Prediction (CTR)

Applying the universal design template to build an ad click-through rate prediction system.

This notebook covers:
1. System design walkthrough for ad CTR prediction
2. Feature engineering (user, ad, context, cross features)
3. Model evolution: LR → GBDT → DNN → DCN/DeepFM
4. Calibration (critical for bidding)
5. Feature crossing techniques
6. Serving at scale (embedding tables, compression, latency)
7. Online learning and explore/exploit
8. Implementation: Deep & Cross Network (DCN) forward pass in PyTorch

---

## Stage 1 — Problem Clarification

**Prompt**: "Design a click-through rate prediction system for online ads."

### Clarifying Questions & Assumed Answers

| Question | Answer |
|----------|--------|
| What type of ads? | Display ads on a content platform (news feed, search results) |
| Scale? | 1B ad impressions/day, ~12K QPS average, ~40K QPS peak |
| Latency? | < 20ms for model inference (within a 100ms auction deadline) |
| Auction type? | Second-price auction, bid = advertiser_bid * predicted_CTR |
| Data? | Billions of (impression, click/no-click) events per day |
| Current solution? | Logistic regression with manual features |
| Goal? | Improve revenue by better CTR prediction → better ad ranking → better auction outcomes |

### Why CTR Prediction Matters

In a second-price CPC (cost-per-click) auction:

$$\text{eCPM} = \text{bid} \times \text{pCTR} \times 1000$$

where eCPM = effective cost per mille (revenue per 1000 impressions).

The ad with the highest eCPM wins the auction. So **accurate CTR prediction directly determines revenue**.

If pCTR is overestimated → bad ads win → users get poor experience → long-term revenue drops.  
If pCTR is underestimated → good ads lose → advertiser ROI is worse → advertisers leave.

---
## Stage 2 — Metrics

### Offline Metrics

| Metric | Why | Notes |
|--------|-----|-------|
| **Log Loss (Binary Cross-Entropy)** | Primary metric — measures calibration | Lower is better. Penalizes confident wrong predictions. |
| **AUC-ROC** | Ranking quality | Does the model rank clicked ads above unclicked? |
| **Calibration (reliability diagram)** | Does predicted 5% CTR mean ~5% actual CTR? | Critical for auction correctness |
| **NE (Normalized Entropy)** | Log loss normalized by baseline entropy | Facebook's preferred metric: NE = LogLoss / H(p), where p is background CTR |

### Why Calibration is King

Unlike recommendations (where ranking is enough), ad systems need **absolute probability estimates**:

- eCPM = bid x pCTR → if pCTR is systematically 2x too high, eCPM is 2x too high → wrong ads win
- Advertiser billing depends on predicted CTR
- Budget pacing (how fast to spend an advertiser's daily budget) depends on calibrated CTR

### Online Metrics

| Metric | What it Measures |
|--------|------------------|
| **Revenue (RPM)** | Revenue per 1000 impressions — the bottom line |
| **eCPM** | Effective cost per mille |
| **CTR** | Actual click-through rate (sanity check) |
| **Advertiser ROI** | Are advertisers getting value? (retention metric) |
| **User engagement** | Does showing better ads improve or hurt user engagement? |

### Guardrail Metrics
- Ad quality score (no spammy/misleading ads)
- User complaints about ads
- Ad load (number of ads per page — keep it reasonable)
- Latency (auction must complete within deadline)

---
## Stage 4 — Feature Engineering

### Feature Taxonomy

#### User Features

| Feature | Type | Notes |
|---------|------|-------|
| user_id | Categorical (high cardinality) | Embedding lookup |
| age_group | Categorical | Bucketed |
| gender | Categorical | |
| interests | Multi-hot | From browsing history |
| historical_CTR | Numerical | User's overall click rate |
| session_depth | Numerical | How deep in session |
| days_since_last_click | Numerical | Recency |

#### Ad Features

| Feature | Type | Notes |
|---------|------|-------|
| ad_id | Categorical (high cardinality) | Embedding lookup |
| advertiser_id | Categorical | |
| ad_category | Categorical | Industry vertical |
| ad_creative_type | Categorical | Image, video, text |
| ad_historical_CTR | Numerical | Ad's overall CTR |
| ad_age_days | Numerical | How long has this ad been running? |
| ad_text_embedding | Dense vector | From ad copy |

#### Context Features

| Feature | Type | Notes |
|---------|------|-------|
| position | Numerical/Categorical | Ad slot position on page |
| device_type | Categorical | Mobile, desktop, tablet |
| os | Categorical | iOS, Android, Windows |
| hour_of_day | Categorical | 0-23 |
| day_of_week | Categorical | 0-6 |
| page_type | Categorical | Feed, search, article |

#### Cross Features (The Secret Sauce)

Feature interactions are critical for CTR prediction. Example:
- `user_gender x ad_category`: Women click on fashion ads more than men
- `hour_of_day x ad_category`: Food delivery ads perform better at lunch/dinner
- `user_interest x ad_category`: Matching interest to ad vertical
- `device_type x ad_creative_type`: Video ads work better on mobile

**Manual crossing** (LR era): explicitly define `gender=F AND category=fashion` as a feature.  
**Learned crossing** (DNN era): let the model learn interactions via embeddings and cross layers.

---
## Stage 5 — Model Evolution

The history of CTR prediction models mirrors the evolution of ML at scale.

### Generation 1: Logistic Regression (2007-2013)

- Features: manual cross features, hashed to a high-dimensional sparse vector
- Training: online SGD with L1 regularization (FTRL-Proximal, Google 2013)
- Pros: Fast training, fast serving, interpretable, handles billions of features
- Cons: Cannot learn complex interactions automatically

### Generation 2: GBDT (2013-2016)

- Facebook's "Practical Lessons" paper (2014): GBDT + LR
  - GBDT learns non-linear transformations of features
  - Each leaf becomes a new binary feature
  - LR on top of GBDT-transformed features
- Pros: Better feature interactions, handles numerical features well
- Cons: Hard to scale to very high cardinality features, hard to do online learning

### Generation 3: DNN-based (2016-present)

- **Wide & Deep** (Google, 2016): Wide (LR on manual crosses) + Deep (DNN on embeddings)
- **DeepFM** (Huawei, 2017): Factorization machines + DNN, learns 2nd-order interactions explicitly
- **DCN** (Google, 2017): Cross network for explicit high-order interactions + DNN
- **DCN-V2** (Google, 2021): Improved cross network with mixture of experts
- **DLRM** (Meta, 2019): Deep Learning Recommendation Model — standard production architecture

### Architecture Pattern (Modern)

```
Sparse features → Embedding tables → Concatenate
Dense features  → Normalize        → Concatenate
                                        ↓
                              [Cross Network] + [Deep Network]
                                        ↓
                                    Combine
                                        ↓
                                   Sigmoid → pCTR
```

---
## Implementation: Deep & Cross Network (DCN) in PyTorch

### Cross Network

The cross network explicitly models feature interactions. At each layer:

$$x_{l+1} = x_0 \odot (W_l x_l + b_l) + x_l$$

where $\odot$ is element-wise multiplication. This creates $(l+1)$-order feature interactions at layer $l$.

**Key insight**: A cross network with $L$ layers models all feature interactions up to order $L+1$, with only $O(d \times L)$ parameters (linear in dimension!), compared to $O(d^L)$ for explicit polynomial features.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class CrossNetwork(nn.Module):
    """
    Cross Network from DCN / DCN-V2.
    
    Each cross layer computes:
        x_{l+1} = x_0 * (W_l @ x_l + b_l) + x_l
    
    This creates explicit feature interactions up to order (num_layers + 1)
    with only O(d * num_layers) parameters.
    """
    
    def __init__(self, input_dim: int, num_layers: int):
        super().__init__()
        self.num_layers = num_layers
        # Each cross layer has a weight matrix and bias
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(input_dim, input_dim) * 0.01)
            for _ in range(num_layers)
        ])
        self.biases = nn.ParameterList([
            nn.Parameter(torch.zeros(input_dim))
            for _ in range(num_layers)
        ])
    
    def forward(self, x0: torch.Tensor) -> torch.Tensor:
        """
        x0: (batch_size, input_dim) — the original input
        Returns: (batch_size, input_dim) — output of cross network
        """
        xl = x0
        for i in range(self.num_layers):
            # x_{l+1} = x_0 * (W_l @ x_l + b_l) + x_l
            cross = x0 * (xl @ self.weights[i].T + self.biases[i])
            xl = cross + xl
        return xl


class DeepNetwork(nn.Module):
    """Standard MLP (the 'Deep' part of DCN)."""
    
    def __init__(self, input_dim: int, hidden_dims: list, dropout: float = 0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.ReLU(),
                nn.BatchNorm1d(dim),
                nn.Dropout(dropout),
            ])
            prev_dim = dim
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)


class DCN(nn.Module):
    """
    Deep & Cross Network (DCN-V2 style).
    
    Architecture:
        Sparse features → Embedding lookup → Concat with dense features
            → Cross Network (explicit interactions)
            → Deep Network (implicit interactions)
            → Combine → Sigmoid → pCTR
    """
    
    def __init__(self,
                 sparse_feature_dims: dict,   # feature_name -> cardinality
                 embed_dim: int = 16,
                 dense_feature_dim: int = 0,
                 cross_layers: int = 3,
                 deep_hidden_dims: list = None,
                 dropout: float = 0.1):
        super().__init__()
        
        if deep_hidden_dims is None:
            deep_hidden_dims = [256, 128, 64]
        
        # Embedding tables for sparse features
        self.embeddings = nn.ModuleDict({
            name: nn.Embedding(cardinality, embed_dim)
            for name, cardinality in sparse_feature_dims.items()
        })
        
        # Total input dimension after concatenation
        total_embed_dim = len(sparse_feature_dims) * embed_dim
        input_dim = total_embed_dim + dense_feature_dim
        
        # Cross network
        self.cross_network = CrossNetwork(input_dim, cross_layers)
        
        # Deep network
        self.deep_network = DeepNetwork(input_dim, deep_hidden_dims, dropout)
        
        # Final output: combine cross and deep outputs
        final_dim = input_dim + deep_hidden_dims[-1]  # cross output + deep output
        self.output_layer = nn.Linear(final_dim, 1)
        
        self.input_dim = input_dim
        self.embed_dim = embed_dim
    
    def forward(self,
                sparse_features: dict,    # feature_name -> (batch_size,) LongTensor
                dense_features: torch.Tensor = None  # (batch_size, dense_dim) FloatTensor
                ) -> torch.Tensor:
        """
        Forward pass.
        Returns: (batch_size, 1) — predicted click probability (logit, before sigmoid)
        """
        # 1. Embedding lookup for sparse features
        embedded = []
        for name, indices in sparse_features.items():
            embedded.append(self.embeddings[name](indices))  # (batch_size, embed_dim)
        
        # 2. Concatenate all embeddings and dense features
        x = torch.cat(embedded, dim=1)  # (batch_size, total_embed_dim)
        if dense_features is not None:
            x = torch.cat([x, dense_features], dim=1)  # (batch_size, input_dim)
        
        # 3. Cross network
        cross_out = self.cross_network(x)  # (batch_size, input_dim)
        
        # 4. Deep network
        deep_out = self.deep_network(x)    # (batch_size, deep_hidden_dims[-1])
        
        # 5. Combine and output
        combined = torch.cat([cross_out, deep_out], dim=1)
        logit = self.output_layer(combined)  # (batch_size, 1)
        
        return logit
    
    def predict_proba(self, sparse_features: dict,
                      dense_features: torch.Tensor = None) -> torch.Tensor:
        """Return predicted CTR probability."""
        logit = self.forward(sparse_features, dense_features)
        return torch.sigmoid(logit)

In [ ]:
# ============================================================
# Demo: Build and run a DCN for ad CTR prediction
# ============================================================

# Define feature schema
sparse_features = {
    "user_id": 10000,        # 10K users
    "ad_id": 5000,           # 5K ads
    "ad_category": 50,       # 50 ad categories
    "device_type": 4,        # mobile, desktop, tablet, other
    "hour_of_day": 24,       # 0-23
    "user_gender": 3,        # male, female, unknown
    "page_type": 5,          # feed, search, article, video, other
}
dense_feature_dim = 5  # user_historical_ctr, ad_historical_ctr, ad_age, session_depth, position

# Build model
model = DCN(
    sparse_feature_dims=sparse_features,
    embed_dim=16,
    dense_feature_dim=dense_feature_dim,
    cross_layers=3,
    deep_hidden_dims=[256, 128, 64],
    dropout=0.1,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
embed_params = sum(p.numel() for name, p in model.named_parameters() if 'embedding' in name)
non_embed_params = total_params - embed_params

print(f"Model architecture: DCN")
print(f"  Sparse features: {len(sparse_features)}")
print(f"  Dense features: {dense_feature_dim}")
print(f"  Embedding dim: 16")
print(f"  Cross layers: 3")
print(f"  Deep layers: [256, 128, 64]")
print(f"\nParameter count:")
print(f"  Embedding params: {embed_params:,}")
print(f"  Network params: {non_embed_params:,}")
print(f"  Total params: {total_params:,}")
print(f"  Model size (FP32): {total_params * 4 / 1e6:.1f} MB")

In [ ]:
# Forward pass with synthetic data
batch_size = 32

# Generate random sparse feature indices
sparse_inputs = {
    name: torch.randint(0, cardinality, (batch_size,))
    for name, cardinality in sparse_features.items()
}

# Generate random dense features
dense_inputs = torch.randn(batch_size, dense_feature_dim)

# Forward pass
model.eval()
with torch.no_grad():
    logits = model(sparse_inputs, dense_inputs)
    probs = torch.sigmoid(logits)

print(f"Input batch size: {batch_size}")
print(f"Output shape: {logits.shape}")
print(f"\nPredicted CTR (first 10 samples):")
for i in range(10):
    print(f"  Sample {i}: pCTR = {probs[i].item():.4f}")

print(f"\nCTR stats: mean={probs.mean():.4f}, std={probs.std():.4f}")
print(f"           min={probs.min():.4f}, max={probs.max():.4f}")

In [ ]:
# ============================================================
# Training loop sketch (educational — showing the key components)
# ============================================================

def train_ctr_model(model, sparse_inputs, dense_inputs, labels,
                    n_epochs=5, lr=0.001, batch_size=256):
    """
    Training loop for CTR prediction.
    Loss: Binary Cross-Entropy (equivalent to log loss).
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()  # numerically stable BCE
    
    n_samples = labels.shape[0]
    model.train()
    
    for epoch in range(n_epochs):
        # Shuffle
        perm = torch.randperm(n_samples)
        total_loss = 0.0
        n_batches = 0
        
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            idx = perm[start:end]
            
            # Batch
            batch_sparse = {k: v[idx] for k, v in sparse_inputs.items()}
            batch_dense = dense_inputs[idx]
            batch_labels = labels[idx]
            
            # Forward
            logits = model(batch_sparse, batch_dense).squeeze(-1)
            loss = criterion(logits, batch_labels)
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        print(f"Epoch {epoch+1}/{n_epochs} | Avg Log Loss: {avg_loss:.4f}")


# Generate synthetic training data
n_train = 10000
train_sparse = {
    name: torch.randint(0, cardinality, (n_train,))
    for name, cardinality in sparse_features.items()
}
train_dense = torch.randn(n_train, dense_feature_dim)
# Synthetic labels: ~5% CTR (realistic for display ads)
train_labels = (torch.rand(n_train) < 0.05).float()

print(f"Training data: {n_train} samples, CTR={train_labels.mean():.3f}")
print()
train_ctr_model(model, train_sparse, train_dense, train_labels, n_epochs=5)

---
## Calibration

### Why Calibration is Critical for Ads

Calibration means: if the model predicts pCTR = 0.05, then among all impressions with pCTR around 0.05, roughly 5% should actually be clicked.

**Why it matters**:
1. **Auction correctness**: eCPM = bid x pCTR. Miscalibrated pCTR → wrong auction winner.
2. **Advertiser billing**: CPC advertisers pay per click. Expected cost = bid x pCTR. If pCTR is wrong, advertisers are over/under-charged.
3. **Budget pacing**: "Spend $1000/day" requires knowing how many impressions to show, which depends on pCTR.

### Calibration Techniques

1. **Platt Scaling**: fit a logistic regression on validation set: $p_{calibrated} = \sigma(a \cdot z + b)$ where $z$ is the model's raw logit.
2. **Isotonic Regression**: non-parametric monotonic calibration on validation set.
3. **Temperature Scaling**: $p_{calibrated} = \sigma(z / T)$ where $T$ is a learned temperature parameter.

### Measuring Calibration

- **Reliability Diagram**: bucket predictions into bins, plot predicted vs actual CTR per bin. Perfect calibration = diagonal line.
- **Expected Calibration Error (ECE)**: weighted average of |predicted - actual| across bins.

In [ ]:
def compute_calibration(predicted_probs, true_labels, n_bins=10):
    """
    Compute calibration metrics.
    
    Returns:
        bin_centers: midpoint of each probability bin
        bin_true_rates: actual positive rate in each bin
        bin_counts: number of samples in each bin
        ece: Expected Calibration Error
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_true_rates = []
    bin_pred_means = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (predicted_probs >= bin_edges[i]) & (predicted_probs < bin_edges[i+1])
        if mask.sum() == 0:
            continue
        bin_centers.append((bin_edges[i] + bin_edges[i+1]) / 2)
        bin_true_rates.append(true_labels[mask].mean())
        bin_pred_means.append(predicted_probs[mask].mean())
        bin_counts.append(mask.sum())
    
    # ECE: weighted average of |predicted_mean - true_rate|
    total_samples = sum(bin_counts)
    ece = sum(
        count / total_samples * abs(pred - true)
        for count, pred, true in zip(bin_counts, bin_pred_means, bin_true_rates)
    )
    
    return bin_centers, bin_true_rates, bin_pred_means, bin_counts, ece


# Demo: show calibration of a simulated model
np.random.seed(42)
n = 50000
true_ctr = 0.05

# Simulated well-calibrated model
pred_good = np.clip(np.random.beta(1.5, 28.5, n), 0.001, 0.999)  # roughly centered around 0.05
labels = (np.random.rand(n) < pred_good).astype(float)

# Simulated poorly-calibrated model (overestimates by 2x)
pred_bad = np.clip(pred_good * 2, 0.001, 0.999)

_, _, _, _, ece_good = compute_calibration(pred_good, labels)
_, _, _, _, ece_bad = compute_calibration(pred_bad, labels)

print(f"Well-calibrated model: ECE = {ece_good:.4f}")
print(f"Poorly-calibrated model (2x overestimate): ECE = {ece_bad:.4f}")

# Show calibration table
centers, true_rates, pred_means, counts, _ = compute_calibration(pred_good, labels)
print(f"\nCalibration table (well-calibrated model):")
print(f"{'Predicted':>10} {'Actual':>8} {'Count':>8}")
for c, t, p, n_val in zip(centers, true_rates, pred_means, counts):
    print(f"{p:>10.4f} {t:>8.4f} {n_val:>8}")

---
## Serving at Scale

### The Embedding Table Challenge

The largest part of a CTR model is the embedding tables:

```
user_id:  100M users x 64 dim x 4 bytes = 25.6 GB
ad_id:    10M ads x 64 dim x 4 bytes = 2.56 GB
Total embedding storage: ~30 GB
```

This doesn't fit in a single GPU! Solutions:

1. **Embedding table partitioning**: shard embeddings across multiple machines
2. **Hashing trick**: hash high-cardinality features to a fixed table size (with collision)
3. **Mixed-dimension embeddings**: frequent items get larger embeddings, rare items get smaller
4. **Embedding compression**: quantize embeddings to INT8 (4x compression)
5. **On-device caching**: cache hot embeddings, fetch cold ones from a distributed store

### Latency Optimization

| Technique | Savings | Tradeoff |
|-----------|---------|----------|
| INT8 quantization | 2-4x speedup | Minor quality loss |
| Model distillation | Custom (student can be much smaller) | Quality loss |
| Feature hashing | Reduces embedding table size | Hash collisions |
| Embedding caching | Reduces lookup latency | Stale embeddings |
| Batched inference | Better GPU utilization | Slight latency increase |
| ONNX Runtime / TensorRT | 2-3x inference speedup | Deployment complexity |

### Auction Latency Budget

```
Ad request received:              0ms
Candidate selection (100 ads):    10ms
Feature lookup (feature store):   15ms
Model inference (100 ads):        15ms
Auction logic:                     5ms
Response:                          5ms
Total:                            ~50ms (well within 100ms budget)
```

---
## Online Learning and Exploration

### Why Online Learning for Ads?

Ad CTR patterns change rapidly:
- New ads appear hourly
- User interests shift with news/events
- Seasonal patterns (holidays, back-to-school)
- Advertiser budget changes

**Daily retraining** misses intra-day patterns. **Online learning** updates the model continuously.

### Online Learning Approaches

1. **FTRL (Follow The Regularized Leader)**: online convex optimization for LR. Google's production choice for years.
2. **Warm-start retraining**: retrain daily, but initialize from previous model weights.
3. **Incremental updates**: update only the embedding tables online, keep DNN weights fixed (retrain DNN daily).

### Exploration vs Exploitation

**Problem**: If we always show the ad with highest predicted CTR, we never learn about new ads.

**Solutions**:

| Strategy | How | Tradeoff |
|----------|-----|----------|
| **Epsilon-greedy** | Show random ad with probability epsilon | Simple, but wastes impressions |
| **Thompson Sampling** | Sample from posterior of CTR estimate | Better exploration, but needs uncertainty |
| **Upper Confidence Bound (UCB)** | Score = predicted_CTR + exploration_bonus | Principled, bonus shrinks with data |
| **Boltzmann exploration** | Sample proportional to exp(score/temperature) | Temperature controls exploration |

In practice: give new ads a small guaranteed impression budget ("exploration budget"), then transition to the model once enough data is collected.

---
## Stage 6-8: Training, Serving, Monitoring Summary

### Training Pipeline

- **Data**: last 7 days of (impression, click) events, sampled to ~1B examples
- **Negative downsampling**: subsample negatives (non-clicks) at rate r, then calibrate predictions by: $p_{calibrated} = \frac{p}{p + (1-p)/r}$
- **Training frequency**: daily retrain + online embedding updates
- **Validation**: temporal split (last day as validation). Never random split for ads data.
- **Distributed training**: data parallel across multiple GPUs, embedding tables sharded

### Serving Architecture

```
Ad Request
    → Ad Retrieval (candidate ads matching targeting criteria)
      → Feature Assembly (feature store lookups)
        → Model Scoring (batch inference on GPU, all candidates at once)
          → Auction (eCPM = bid * pCTR, second-price)
            → Ad Response
```

### Monitoring

| What | How | Alert |
|------|-----|-------|
| Calibration | Daily reliability diagram, ECE | ECE > 0.02 |
| Revenue | Hourly revenue vs forecast | > 10% below forecast |
| CTR | Hourly actual CTR vs predicted | > 20% deviation |
| Feature coverage | Null/default rate per feature | > 5% null on key feature |
| Latency | p50, p99 scoring latency | p99 > 20ms |
| Model freshness | Time since last model update | > 2 hours for online model |

---
## Business Metrics Deep Dive

### Revenue

$$\text{Revenue} = \sum_{\text{impressions}} \text{CPC} \times \mathbb{1}[\text{click}]$$

In second-price auction:
$$\text{CPC}_{\text{winner}} = \frac{\text{eCPM}_{\text{2nd place}}}{\text{pCTR}_{\text{winner}} \times 1000}$$

So the winning advertiser pays just enough to beat the second-place bid.

### eCPM (Effective Cost Per Mille)

$$\text{eCPM} = \text{bid} \times \text{pCTR} \times 1000$$

Higher eCPM = more revenue per impression slot.

### Advertiser ROI

$$\text{ROI} = \frac{\text{conversion\_value} - \text{ad\_spend}}{\text{ad\_spend}}$$

If advertiser ROI is negative, they'll leave the platform. Long-term revenue depends on keeping advertisers happy.

### The Three-Sided Marketplace

- **Users** want relevant, non-intrusive ads
- **Advertisers** want high ROI
- **Platform** wants maximum revenue

A good CTR model serves all three: better predictions → more relevant ads → happier users → higher CTR → better advertiser ROI → more ad spend → more platform revenue.

---
## Key Interview Talking Points

1. **Calibration is king**: unlike recs/search, ads need absolute probability estimates. Always discuss calibration.
2. **Know the auction**: second-price, eCPM = bid x pCTR. This is why pCTR accuracy matters.
3. **DCN/DeepFM for feature interactions**: know how the cross network works and why it's better than manual feature crossing.
4. **Embedding tables dominate model size**: discuss sharding, hashing, compression.
5. **Negative downsampling + calibration correction**: standard technique for handling billions of mostly-negative examples.
6. **Online learning for freshness**: ads change fast. Discuss FTRL or incremental embedding updates.
7. **Exploration is necessary**: new ads need impressions to estimate CTR. Discuss Thompson sampling or exploration budgets.
8. **Business metrics tie it together**: revenue, eCPM, advertiser ROI. Show you understand the business, not just the model.
9. **Position bias**: users click position 1 more than position 5 regardless of relevance. Discuss position features or causal debiasing.